## **Deutsche Version**

### **Datenmodellierung**

Wir haben drei Tabellen in diesem Beispiel:

1. **Traders (Händler)**  
   - Enthält Informationen über jeden Händler  
   - Spalten:  
     - `trader_id`: eindeutige ID für jeden Händler  
     - `name`: Name des Händlers  

2. **Stocks (Aktien)**  
   - Enthält Informationen über Aktien  
   - Spalten:  
     - `symbol`: Börsenticker der Aktie  
     - `description`: Beschreibung der Aktie  

3. **Trades (Transaktionen)**  
   - Enthält alle Handelsaktivitäten  
   - Spalten:  
     - `trade_id`: eindeutige Trade-ID  
     - `trader_id`: referenziert einen Händler (1:n Beziehung)  
     - `symbol`: referenziert eine Aktie (1:n Beziehung)  
     - `quantity`: Anzahl gehandelter Aktien  

**Beziehungen:**  
- Jeder Händler kann mehrere Trades haben → **1:n** (Händler → Trades)  
- Jeder Trade bezieht sich auf eine Aktie → **1:n** (Aktie → Trades)  

---

### **Aufgabenstellung**

Deine Aufgabe ist es, **die Daten zu analysieren** und **aggregierte Ergebnisse** zu erstellen:

1. Führe die Tabellen zusammen, sodass du **Händlernamen, Aktienbeschreibung und Handelsmengen** in einem DataFrame hast.  
2. Berechne folgende **aggregierte Spalten pro Händler**:
   - Gesamtmenge aller Trades  
   - Durchschnittliche Menge pro Trade  
   - Anzahl der Trades  
   - Optional: Gesamtmenge bestimmter Aktien (z. B. AAPL + TSLA)  
3. Bonus: Finde **welcher Händler die meisten Aktien jeder Aktie gehandelt hat**.  

Verwende **pandas-Funktionen** wie `merge`, `groupby`, `sum`, `mean` und `count`.  

### **Erwartetes Ergebnis**

Das finale DataFrame sollte wie folgt aussehen:

| name    | total_quantity | avg_quantity | num_trades | AAPL_TSLA_total |
|---------|----------------|--------------|------------|----------------|
| Alice   | 30             | 15.0         | 2          | 30.0           |
| Bob     | 15             | 7.5          | 2          | 0.0            |
| Charlie | 15             | 15.0         | 1          | 0.0            |

In [1]:
import pandas as pd

# Traders
traders = pd.DataFrame({
    "trader_id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"]
})

# Stocks
stocks = pd.DataFrame({
    "symbol": ["AAPL", "GOOG", "TSLA", "MSFT", "AMZN"],
    "description": [
        "Apple Inc.",
        "Alphabet Inc.",
        "Tesla Inc.",
        "Microsoft Corp.",
        "Amazon.com Inc."
    ]
})

# Trades
trades = pd.DataFrame({
    "trade_id": [101, 102, 103, 104, 105],
    "trader_id": [1, 2, 1, 3, 2],
    "symbol": ["AAPL", "GOOG", "TSLA", "MSFT", "AMZN"],
    "quantity": [10, 5, 20, 15, 10]
})

# Merge tables
merged = trades.merge(traders, on="trader_id").merge(stocks, on="symbol")

# 1. Total quantity per trader
agg_trader = merged.groupby("name")["quantity"].sum().reset_index()
agg_trader.rename(columns={"quantity": "total_quantity"}, inplace=True)

# 2. Average quantity per trade
agg_trader["avg_quantity"] = merged.groupby("name")["quantity"].mean().values

# 3. Number of trades per trader
agg_trader["num_trades"] = merged.groupby("name")["trade_id"].count().values

# 4. Example: total quantity of AAPL + TSLA per trader
# First filter relevant symbols
filtered = merged[merged["symbol"].isin(["AAPL", "TSLA"])]
agg_trader["AAPL_TSLA_total"] = filtered.groupby("name")["quantity"].sum().reindex(agg_trader["name"]).fillna(0).values

print(agg_trader)

      name  total_quantity  avg_quantity  num_trades  AAPL_TSLA_total
0    Alice              30          15.0           2             30.0
1      Bob              15           7.5           2              0.0
2  Charlie              15          15.0           1              0.0
